<a href="https://colab.research.google.com/github/Gabrielt2503/Computational-Linguistics/blob/main/tutorials/Tutorial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 1: Language, Text, and Corpus Inspection

This is the first tutorial with practical exercises for the lecture *Introduction to Computational Linguistics* in the Winter Semester *2026*. Hands-on exercises are marked with 👋 ⚒ and questions are marked with ❓.

## **Store this notebook**

Go to **File** and make sure you store this file as a local copy to either your GitHub or Google Drive. If you do not have a Google account and also do not want to create one, please check Option C below.

## Option A)  Google Drive

Go to the menu **File** and click on **Save a Copy in Drive**. The advantage of this option is that your changes in the notebook will be saved regularly. If you wish to move the file to a specific folder, click on **Move** and the select the corresponding folder in your Google Drive.

## Option B) GitHub


Go to the menu **File** and click on **Save a Copy in GitHub**, select the repository on your GitHub account to which you wish to push the Colab notebook. Note that this option does not automatically save changes in your Colab notebook.

## Option C) Download this notebook as ipynb (Jupyter notebook) or py (Python file)

To run this notebook as Jupyter notebook or Python file on your local machine, Python and NLTK need to be installed. Within the course of the semester several more local installations will be required.  

To run a Jupyter notebook the easiest option is to install [Anacadonda](https://www.anaconda.com/). A Python file can be run directly from the command line or in an Integrated Development Environment (IDE), such as [PyCharm](https://www.jetbrains.com/de-de/pycharm/).

-----------

## **Part 1: Text Encoding**

When comparing sequences or texts, especially multilingual ones, we need to make sure that the encoding is the same.

👋 ⚒ Exercise 1: Use the function `unicode.name()` as shown here to run through both strings `a` and `b` in parallel and print each position of a character next to each other.

The output should be:
```
a: ['LATIN SMALL LETTER N'] b: ['LATIN SMALL LETTER N']
a: ['LATIN SMALL LETTER A'] b: ['LATIN SMALL LETTER A']
...
```


In [9]:
import unicodedata

a = "naïve"
b = "nai\u0308ve"

print(a, b)
print(len(a), len(b))
print(a == b)

for char_a, char_b in zip(a + " ", b):
  print(f"a: {[unicodedata.name(char_a,)]!s:<44} b: {[unicodedata.name(char_b)]}")

naïve naïve
5 6
False
a: ['LATIN SMALL LETTER N']                     b: ['LATIN SMALL LETTER N']
a: ['LATIN SMALL LETTER A']                     b: ['LATIN SMALL LETTER A']
a: ['LATIN SMALL LETTER I WITH DIAERESIS']      b: ['LATIN SMALL LETTER I']
a: ['LATIN SMALL LETTER V']                     b: ['COMBINING DIAERESIS']
a: ['LATIN SMALL LETTER E']                     b: ['LATIN SMALL LETTER V']
a: ['SPACE']                                    b: ['LATIN SMALL LETTER E']


If we wish to make a and b directly comparable, we can use another function already provided by the unicodedata library.  "NFC" is short for Normalization Form Canonical Composition.

In [10]:
print(unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC",b))
for char_b in unicodedata.normalize("NFC",b):
  print(f"b: {[unicodedata.name(char_b)]}")

True
b: ['LATIN SMALL LETTER N']
b: ['LATIN SMALL LETTER A']
b: ['LATIN SMALL LETTER I WITH DIAERESIS']
b: ['LATIN SMALL LETTER V']
b: ['LATIN SMALL LETTER E']


When it comes to characters in multiple languages it is good to be able to check the encoding.

❓ Question 1: What happens to the characters below? What do we need to pay attention to when comparing words?

In [11]:
print("ß".upper(), len("ß".upper()), len("ß"))   # German sharp s
print("İ".lower(), len("İ".lower()), len("İ"))   # Turkish dotted capital I
print(len("你好吗"))                              # Chinese
family = "\U0001F468\u200D\U0001F469\u200D\U0001F467"   # Unicode code for family emoji
print(family, len(family))

SS 2 1
i̇ 2 1
3
👨‍👩‍👧 5


👋 ⚒ Exercise 2: Write a short function that automatically compares the length of two strings based on their visible characters, the way humans see them.

🌶 Advanced option: make sure that the function also returns a count of 1 for the family emoji. The emoji consists of three people joined by the zero-width joiner `\u200d` that is used to indicate unicode characters are to be joined.  

In [12]:
def count_visible_chars(text):
  count = 0   # How many variables counted so far
  # Your code here
  join_next = False # it says "did we find a zero-width joiner?" NO (False), so we count the next character
  text = unicodedata.normalize("NFC", text) # converting the string to a NFC ( more characters into 1 like e and è)
  for c in text:  # for each character in text...
    if c == "\u200d": # if the current character is a joiner...
      join_next = True # We found zero-width joiner, so we skip counting because character belongs to the previous emoji
    else: # if it is not a joiner...
      if not join_next: # if it s NO (False), count teh character
        count += 1 # add 1 to the counter
      join_next = False # reset for the next character
  return count # return the final count

count_visible_chars(family)

1

## **Part 2: From Plural to Singular**

Before we look into existing libraries that effectively turn words into their base forms, we will try to write our own tiny rule-based function that can turn plural nouns into singular nouns.

👋 ⚒ Exercise 3: Start your function with the first rule of removing the "s" for regular cases. Then add more rules to cover more cases. What to do with the completely irregular words? Count how many words from the `plural_tests` dictionary your function converted correctly as absoulte  and relative count. Relative count is a percentage that shows the accuracy of your function.


In [13]:
irregulars = { "mice": "mouse", "children": "child", "lives": "life",
    "sheep": "sheep", "news": "news",}

def from_plural_to_singular(word):
    # Your code here#
    if word in irregulars.keys():
      return irregulars[word]
    elif word.endswith("ies"):
      return word.replace("ies", "y")
    elif word.endswith("es"):
      return word.replace("es", "")
    elif word.endswith("s"):
      return word.replace("s", "")


# This dictionary contains the plural noun and the correct singular version to compare the result of your function to
plural_tests = {
    "cats": "cat", "days": "day", "boys": "boy",
    "buses": "bus", "boxes": "box", "wishes": "wish", "churches": "church",
    "cities": "city",
    "mice": "mouse", "children": "child", "lives": "life",
    "sheep": "sheep", "news": "news",
}



correct = 0
for key, value in plural_tests.items():
  singular = from_plural_to_singular(key)
  if singular == value:
    correct += 1
  print(key, singular)
print()




cats cat
days day
boys boy
buses bus
boxes box
wishes wish
churches church
cities city
mice mouse
children child
lives life
sheep sheep
news news



## **Part 3: Ambiguity**

Many words have more than one word class called part of speech. Next week, we will look at Part-of-Speech (PoS) tagging and existing libraries. Today, we will give a simple hand-made lexicon version a try.

The following cell defines our lexicon and our trial sentences. Let's run this first.


In [14]:
lexicon = {
    "time":      ["NOUN", "VERB"],
    "flies":     ["NOUN", "VERB"],
    "like":      ["VERB", "ADP"],
    "an":        ["DET"],
    "a":         ["DET"],
    "the":       ["DET"],
    "arrow":     ["NOUN"],
    "fruit":     ["NOUN"],
    "banana":    ["NOUN"],
    "i":         ["PRON"],
    "saw":       ["NOUN", "VERB"],
    "her":       ["PRON", "DET"],
    "duck":      ["NOUN", "VERB"],
    "man":       ["NOUN", "VERB"],
    "with":      ["ADP"],
    "telescope": ["NOUN"],
}

sentences = [
    "time flies like an arrow",
    "fruit flies like a banana",
    "i saw her duck",
    "i saw the man with the telescope",
]

For the following exercise it is useful to work with tuples. Tuples are immutable arrays that are written in parenthesis and lists are written in square brackets.

In [15]:
pos_tags = ("NOUN", "VERB", "DETERMINER", "PRONOUN")
print(pos_tags)

('NOUN', 'VERB', 'DETERMINER', 'PRONOUN')


Elements can be accessed via the index and you can also get the length of the tuple.

In [16]:
print(pos_tags[0])
print(len(pos_tags))

NOUN
4


Tuples do not support item re-assignment as lists do

In [17]:
pos_tags[0] = "Adjective"

TypeError: 'tuple' object does not support item assignment

Since it is not possible to directly add elements to a tuple, you need to create a new tuple with the new element. This is similar to combining two tuples where the second one only has the single element "ADJECTIVES". We need to write the comma to show it is a tuple.

In [18]:
pos_tags_extended = pos_tags + ("ADJECTIVE",)
print(pos_tags_extended)

('NOUN', 'VERB', 'DETERMINER', 'PRONOUN', 'ADJECTIVE')


👋 ⚒ Exercise 4: Write a function that reads all sentences in the variable `sentences` and returns all possible part-of-speech sequences for the sentences that are stored in the lexicon. Return a list that contains tuples, where each tuple is one possible part-of-speech sequence of the sentence.

In [21]:
def pos_tagging_sequences(sentence):
    sequences = [()]

    # Your code here
    """for word in sentence.split():
      new_sequences = []
      for seq in sequences:
        for tag in lexicon[word]:
          new_sequences.append(seq + (tag,))
      sequences = new_sequences
    return sequences"""
    words = sentence.split()
    for word in words:
      interim_sequence = []
      for seq in sequences:
        for pos_tag in lexicon.get(word):
          interim_sequence.append(seq+(pos_tag,))
      sequences = interim_sequence
    return sequences


for s in sentences:
    tagged_s = pos_tagging_sequences(s)
    print(tagged_s, "sequences:", s)

[('NOUN', 'NOUN', 'VERB', 'DET', 'NOUN'), ('NOUN', 'NOUN', 'ADP', 'DET', 'NOUN'), ('NOUN', 'VERB', 'VERB', 'DET', 'NOUN'), ('NOUN', 'VERB', 'ADP', 'DET', 'NOUN'), ('VERB', 'NOUN', 'VERB', 'DET', 'NOUN'), ('VERB', 'NOUN', 'ADP', 'DET', 'NOUN'), ('VERB', 'VERB', 'VERB', 'DET', 'NOUN'), ('VERB', 'VERB', 'ADP', 'DET', 'NOUN')] sequences: time flies like an arrow
[('NOUN', 'NOUN', 'VERB', 'DET', 'NOUN'), ('NOUN', 'NOUN', 'ADP', 'DET', 'NOUN'), ('NOUN', 'VERB', 'VERB', 'DET', 'NOUN'), ('NOUN', 'VERB', 'ADP', 'DET', 'NOUN')] sequences: fruit flies like a banana
[('PRON', 'NOUN', 'PRON', 'NOUN'), ('PRON', 'NOUN', 'PRON', 'VERB'), ('PRON', 'NOUN', 'DET', 'NOUN'), ('PRON', 'NOUN', 'DET', 'VERB'), ('PRON', 'VERB', 'PRON', 'NOUN'), ('PRON', 'VERB', 'PRON', 'VERB'), ('PRON', 'VERB', 'DET', 'NOUN'), ('PRON', 'VERB', 'DET', 'VERB')] sequences: i saw her duck
[('PRON', 'NOUN', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN'), ('PRON', 'NOUN', 'DET', 'VERB', 'ADP', 'DET', 'NOUN'), ('PRON', 'VERB', 'DET', 'NOUN', '

👋 ⚒ Exercise 5: Write rules that can be used to eliminate specific POS sequences, such as


*   The sequence has a VERB
*   That the sequence has exactly one verb
*   A determiner DET is followed by a noun
*   The sequence has no directly adjacent nouns

Use the library re for pattern matching for this purpose. One example is provided.

In [22]:
import re

def no_adjacent_nouns(tags):
    return re.search(r"NOUN NOUN", " ".join(tags)) is None

example_tag_sequence = ("NOUN", "NOUN", "VERB", "DET")

print(no_adjacent_nouns(example_tag_sequence))

def has_verb(tags):
  return re.search(r"VERB", " ".join(tags)) is not None
example_tag_sequence = ("VERB",)
print(has_verb(example_tag_sequence))

def exactly_one_verb(tags):
  return tags.count("VERB") == 1
example_tag_sequence = ("VERB",)
print(exactly_one_verb(example_tag_sequence))

def det_followed_by_noun(tags):
  return re.search(r"DET NOUN", " ".join(tags)) is not None
example_tag_sequence = ("DET", "NOUN")
print(det_followed_by_noun(example_tag_sequence))


False
True
True
True
